# Data Splitting Notebook

This notebook splits the engineered feature dataset (`df_final`) into training, validation, test, and production datasets according to the following distribution:

- Training: ~40%
- Validation: ~10%
- Testing: ~10%
- Production (holdout): ~40%

The goal is to ensure balanced evaluation and reserve unseen data for production simulations.


In [4]:
# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
import boto3

In [6]:
import boto3

s3 = boto3.client('s3')
bucket = 'sagemaker-us-east-1-520193275417'
prefix = 'final_project/engineered/'

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
for obj in response.get('Contents', []):
    print(obj['Key'])


final_project/engineered//engineered_features.csv


In [8]:
df_final = pd.read_csv('s3://sagemaker-us-east-1-520193275417/final_project/engineered//engineered_features.csv')
df_final.head()


,label,subflow_fwd_bytes,avg_fwd_segment_size,fwd_packet_length_max,fwd_act_data_packets,fwd_packet_length_mean,fwd_iat_std,fwd_packets_length_total,init_fwd_win_bytes,subflow_fwd_packets,fwd_header_length,record_id,event_time
0,0,12,6.0,6,1,6.0,0.0,12,33,2,40,7decd078-583a-4d1f-824b-1dfab903fe53,1.749057e+09
1,0,6,6.0,6,0,6.0,0.0,6,29,1,20,08d9e0c3-0a2d-4298-889c-b1a5a2a4b398,1.749057e+09
2,0,6,6.0,6,0,6.0,0.0,6,29,1,20,e18e086d-ca51-45f2-8917-2855a05633e6,1.749057e+09
3,0,6,6.0,6,0,6.0,0.0,6,31,1,20,77742f01-3581-47c1-941d-9fdc9bdb0d64,1.749057e+09
4,0,12,6.0,6,1,6.0,0.0,12,32,2,40,866cd496-da04-4b3e-a521-8f5aa31bcb96,1.749057e+09


In [9]:
# Assuming df_final is already loaded or imported
# Split into features and labels
X = df_final.drop('label', axis=1)
y = df_final['label']

# Reserve 40% for production
X_temp, X_prod, y_temp, y_prod = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

# Split the remaining 60% into train (~67%), val (~17%), and test (~17%)
X_train, X_temp2, y_train, y_temp2 = train_test_split(X_temp, y_temp, test_size=0.33, random_state=42, stratify=y_temp)
X_val, X_test, y_val, y_test = train_test_split(X_temp2, y_temp2, test_size=0.5, random_state=42, stratify=y_temp2)

# Output summary
print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")
print(f"Production size: {len(X_prod)}")


Train size: 88947
Validation size: 21905
Test size: 21906
Production size: 88506


# Storing Data Split in S3 Bucket 

## Saving Data Split CSV files

In [12]:
# Create local CSVs for each dataset split

# Combine features and labels for each split
train_df = pd.concat([X_train, y_train], axis=1)
val_df = pd.concat([X_val, y_val], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)
prod_df = pd.concat([X_prod, y_prod], axis=1)

# Save to CSV
train_df.to_csv("Xy_train.csv", index=False)
val_df.to_csv("Xy_val.csv", index=False)
test_df.to_csv("Xy_test.csv", index=False)
prod_df.to_csv("Xy_prod.csv", index=False)

print("All split datasets saved locally.")


All split datasets saved locally.


In [13]:
# Initialize S3 client
s3 = boto3.client('s3')

# Define bucket and S3 prefix path
bucket_name = 'sagemaker-us-east-1-520193275417'
s3_prefix = 'final_project/feature_engineer/'

# Define local file paths and S3 target keys
files_to_upload = {
    "Xy_train.csv": f"{s3_prefix}Xy_train.csv",
    "Xy_val.csv": f"{s3_prefix}Xy_val.csv",
    "Xy_test.csv": f"{s3_prefix}Xy_test.csv",
    "Xy_prod.csv": f"{s3_prefix}Xy_prod.csv"
}

# Upload loop
for local_file, s3_key in files_to_upload.items():
    s3.upload_file(local_file, bucket_name, s3_key)
    print(f"Uploaded {local_file} to s3://{bucket_name}/{s3_key}")


Uploaded Xy_train.csv to s3://sagemaker-us-east-1-520193275417/final_project/feature_engineer/Xy_train.csv
Uploaded Xy_val.csv to s3://sagemaker-us-east-1-520193275417/final_project/feature_engineer/Xy_val.csv
Uploaded Xy_test.csv to s3://sagemaker-us-east-1-520193275417/final_project/feature_engineer/Xy_test.csv
Uploaded Xy_prod.csv to s3://sagemaker-us-east-1-520193275417/final_project/feature_engineer/Xy_prod.csv
